# User Behavior Analysis

用数据回答：哪些用户贡献了记录？他们更常谈什么？公开文本中出现了哪些情绪词？

输入为 `tutorials/data/posts_two_weeks.parquet` 中四川的可用记录。
源记录不是平台全部发帖，也不是完整用户历史。
城市作为来源标签分组；日期用于确定两周窗口和计算观测到发帖的天数。

### 分析哪些行为？

分析对象是已采集记录中的发帖与公开表达，以及这些记录在城市来源标签上的分布。

| 维度 | 具体观察 | 指标解释 |
|---|---|---|
| 发帖数量 | 两周记录数、出现日期数、记录贡献比例 | 窗口内采集到的发帖记录量 |
| 发帖内容 | 工作、学习、饮食、休闲、出行话题、公共事务词的命中比例 | 记录中出现这些话题词的比例 |
| 公开情绪表达 | 正向词、负向词、同时命中、未命中 | 公开文本中的情绪词命中情况 |
| 社交互动 | 转发、@提及、点赞、评论 | 先检查是否有可验证的结构化字段 |
| 城市汇总 | 各来源城市记录数、用户数、每用户平均记录数 | 城市标签下采集记录的分布 |

谈论饮食不一定表示喜欢饮食，提到出行不等于实际出行，情绪词也不等于用户的心理状态。
每个指标都需要明确分析单位、分母和解释范围。

### 目录

- [1．数据能告诉我们什么？](#data)
- [2．哪些用户贡献了记录？](#activity)
- [3．用户更常谈哪些内容？](#content)
- [4．公开表达中的情绪词](#expression)
- [5．城市汇总：数量与每用户记录数](#city)
- [6．开放问题](#questions)

<a id="data"></a>
## 1．数据能告诉我们什么？

### 1.1 环境和真实输入

在 VS Code 打开整个项目，选择课程 Conda 内核。本教程独立初始化，
不依赖其他 Notebook 的变量，只用现有 Pandas、Matplotlib 与 PyArrow 依赖。
输出表格与图显示在 Notebook 中。

In [ ]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import PercentFormatter
from IPython.display import display

assert (Path(sys.prefix) / 'conda-meta').exists(), '请在右上角选择 Conda 内核'
TUTORIAL_DIR = Path('.') if Path('data/posts_two_weeks.parquet').is_file() else Path('tutorials')
DATA = TUTORIAL_DIR / 'data/posts_two_weeks.parquet'
assert DATA.is_file(), '请从 tutorials 或项目根目录启动内核，并保留 tutorials/data'
ROOT = TUTORIAL_DIR.resolve().parent
source_columns = pq.ParquetFile(DATA).schema_arrow.names
raw = pd.read_parquet(DATA,
    columns=['post_id', 'user_id', 'date', 'city', 'text',
             'sentiment_label_source', 'score_valid'],
    filters=[('province_code', '==', '51'), ('usable_text', '==', True)])
posts = raw.copy()

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
CJK_FONT = next((name for name in ['Noto Sans CJK SC', 'Arial Unicode MS',
    'Microsoft YaHei', 'SimHei', 'Songti SC', 'Heiti TC', 'Noto Sans CJK JP']
    if name in available_fonts), None)
if CJK_FONT:
    plt.rcParams['font.family'] = CJK_FONT
plt.rcParams.update({'font.size': 11, 'figure.figsize': (9, 4),
    'axes.unicode_minus': False, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 8)
pd.set_option('display.max_colwidth', 90)

def label(chinese, english):
    return chinese if CJK_FONT else english

WINDOW_DAYS = 14
print('Pandas：', pd.__version__, 'Python：', sys.executable)
print('帖子记录数：', len(posts), '不同用户数：', posts['user_id'].nunique(),
      '出现日期数：', posts['date'].nunique())
display(posts[['date', 'city', 'text']].head(3))

**怎么看结果：** 默认范围有 54,609 条可用记录、36,463 个匿名用户，窗口为 2022-08-12 至 08-25。
`post_id` 是帖子主键，`user_id` 用于把同一用户的记录组合起来。
用户不是逐行独立的：同一个人的多条帖子可能相关，不能在统计推断时直接当成独立样本。

### 1.2 能不能分析转发、@提及或点赞？先查字段

一个平台可能支持很多行为，但一个数据文件不一定保留这些行为的字段。
本文件没有独立的原创/转发标记、转发父帖 ID、被提及账号列表或互动数量。
文本来自预处理的 `text_clean`，随后又做了脱敏；即使某个文本模式看不见，也不能断言原帖没有对应行为。

In [ ]:
interaction_fields = ['is_repost', 'retweeted_status_id', 'mention_ids',
                       'reposts_count', 'likes_count', 'comments_count']
field_audit = pd.DataFrame({
    '示意字段名': interaction_fields,
    '当前文件是否存在': [name in source_columns for name in interaction_fields],
})
display(field_audit)
text_audit = pd.Series({
    '正文含 @ 字符的记录数': posts['text'].str.contains('@', regex=False, na=False).sum(),
    '正文含 [提及] 掩码的记录数': posts['text'].str.contains('[提及]', regex=False, na=False).sum(),
    '正文含“转发”一词的记录数': posts['text'].str.contains('转发', regex=False, na=False).sum(),
}, name='可见文本计数')
display(text_audit.to_frame())

**怎么看结果：** 本范围里没有可见的 `@` 或 `[提及]`；有 21 条正文包含“转发”一词。
**21 不是转发帖数，0 也不是用户从未提及别人。**
提到“转发”可以是在讨论这个动作，不能替代平台转发元数据。
将来有可验证的动作类型、被提及账号 ID 和转发父帖 ID，才适合分析这些互动或关系。
本教程因此把记录统一称为“帖子记录”，不把它们全部判为原创，也不估计传播影响力。
字段说明见项目 `tutorials/data/README.md`，脱敏过程见 `scripts/prepare_two_weeks.py`。

<a id="activity"></a>
## 2．哪些用户贡献了记录？

### 2.1 从帖子表变成用户表

先定义指标，再选聚合方式。每个用户一行可以方便地比较用户之间的差异。

| 字段 | 定义 | 分析单位 |
|---|---|---|
| `n_posts` | 该用户在窗口内的采集记录数 | 用户 |
| `observed_days` | 有采集记录的不同日期数 | 用户 |
| `observed_day_share` | 观测日期数 / 14 个日历日 | 用户 |
| `mean_text_length` | 该用户已采集正文的平均字符数 | 用户 |

`observed_day_share` 描述窗口内观测到的发帖日期覆盖，不等于每天使用微博的概率。
字符数也不是词数。没有采集记录的日期，不能直接解释为用户没有使用微博。

In [ ]:
posts['text_length'] = posts['text'].str.len()
users = posts.groupby('user_id').agg(
    n_posts=('post_id', 'size'),
    observed_days=('date', 'nunique'),
    mean_text_length=('text_length', 'mean'),
)
users['observed_day_share'] = users['observed_days'] / WINDOW_DAYS
display(users.head(3))
display(users[['n_posts', 'observed_days', 'mean_text_length']].describe())
print('用户表累计记录数：', users['n_posts'].sum(), '原帖子表记录数：', len(posts))

### 2.2 均值和中位数分别回答什么？

均值反映所有用户的记录总量除以人数；中位数反映排序后处于中间位置的用户。
若少数用户有很多记录，均值可能大于多数用户的记录数。
先看分位数和记录数分布，再决定怎样描述“典型用户”。

In [ ]:
summary = pd.Series({
    '用户数': len(users),
    '每用户平均记录数': users['n_posts'].mean(),
    '每用户记录数中位数': users['n_posts'].median(),
    '记录数75%分位数': users['n_posts'].quantile(0.75),
    '记录数95%分位数': users['n_posts'].quantile(0.95),
    '最大记录数': users['n_posts'].max(),
}, name='统计值')
display(summary.to_frame())
display(users['n_posts'].value_counts().sort_index().head(8).rename('用户数').to_frame())

**怎么看结果：** 当前窗口中多数用户只被观测到一条帖子，记录数中位数为 1。
这描述样本的稀疏程度，不能据此把他们称为平台上的“低活跃用户”。

### 2.3 用户占比和记录贡献占比的分母不同

将用户按采集记录数划为互斥组：1 条、2 条、3–4 条、5 条及以上。
一种比例数人，另一种比例数帖；两者都要报告。

In [ ]:
groups = pd.cut(users['n_posts'], bins=[0, 1, 2, 4, np.inf],
                labels=['1', '2', '3–4', '5+'])
group_summary = users.assign(record_group=groups).groupby(
    'record_group', observed=False).agg(n_users=('n_posts', 'size'), n_posts=('n_posts', 'sum'))
group_summary['user_share'] = group_summary['n_users'] / len(users)
group_summary['post_share'] = group_summary['n_posts'] / len(posts)
display(group_summary)
plot_groups = group_summary[['user_share', 'post_share']].rename(columns={
    'user_share': label('用户占比', 'User share'), 'post_share': label('记录贡献占比', 'Post share')})
ax = plot_groups.plot.bar(rot=0, color=['#2563a6', '#c56a28'])
ax.set(title=label('各记录数组：人数与记录贡献', 'Users and post contribution by record count'),
       xlabel=label('窗口采集记录数组', 'Collected posts per user in window'),
       ylabel=label('比例', 'Proportion'))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

**怎么看结果：** 一组人数较少，也可能贡献相当一部分帖子。
分组门槛是分析者选的定义，不是自然存在的用户类型。下面的开放问题会比较不同门槛。

<a id="content"></a>
## 3．用户更常谈哪些内容？

### 3.1 把内容问题变成可复核的关键词规则

不直接给用户贴“职业”“爱好”标签，而是先统计记录中的词命中。
词表沿用主课的六个讨论维度，便于在教程和实验之间切换。
每条帖子可以同时命中多类，因此六类比例相加不一定等于 1。

In [ ]:
TOPICS = {
    '工作': ['工作', '上班', '加班', '公司', '同事', '办公'],
    '学习': ['学习', '考试', '作业', '学校', '大学', '论文'],
    '饮食': ['吃饭', '好吃', '火锅', '奶茶', '美食', '早餐', '午饭', '晚饭'],
    '休闲': ['电影', '音乐', '游戏', '演唱会', '追剧', '公园'],
    '出行话题': ['旅行', '旅游', '出差', '高铁', '航班', '地铁', '公交'],
    '公共事务': ['停电', '限电', '停水', '疫情', '政府', '社区', '防疫', '核酸'],
}

def keyword_hit(text_series, words):
    # 转义词中的正则字符；| 表示命中任意一个词。
    pattern = '|'.join(re.escape(word) for word in words)
    return text_series.str.contains(pattern, regex=True, na=False)

topic_flags = pd.DataFrame({name: keyword_hit(posts['text'], words)
                            for name, words in TOPICS.items()}, index=posts.index)
display(pd.DataFrame({'主题': TOPICS.keys(), '词表': ['、'.join(w) for w in TOPICS.values()]}))
topic_summary = pd.DataFrame({'命中帖子数': topic_flags.sum(), '帖子命中比例': topic_flags.mean()})
display(topic_summary.sort_values('命中帖子数', ascending=False))
print('六类中至少命中一类的帖子比例：', topic_flags.any(axis=1).mean())
print('六类全部未命中的帖子数：', (~topic_flags.any(axis=1)).sum())

“出行话题”只是讨论内容，未使用坐标判断真实出行。
`str.contains` 是子串匹配，不是中文分词或语义模型。例如“公司”也可能出现在转述、新闻或广告中。
多标签允许重叠，但未命中的记录仍必须留在相应分母中。

### 3.2 同一词表，帖子等权与用户等权

对所有帖子求均值，每条帖子权重相同，高记录用户会贡献更多。
先对每个用户求命中比例，再对用户求均值，每位已观测用户权重相同。
二者都合理，但回答不同的问题。

In [ ]:
topic_user_share = topic_flags.groupby(posts['user_id']).mean()
users = users.join(topic_user_share.add_prefix('topic_share_'))
topic_weighting = pd.DataFrame({
    'post_weighted': topic_flags.mean(),
    'user_weighted': topic_user_share.mean(),
})
display(topic_weighting.rename(columns={
    'post_weighted': '帖子等权比例', 'user_weighted': '用户等权比例'}))
ax = topic_weighting.rename(index={
    k: label(k, v) for k, v in zip(TOPICS, ['Work', 'Study', 'Food', 'Leisure', 'Travel topics', 'Public affairs'])
}).rename(columns={'post_weighted': label('帖子等权', 'Posts equally weighted'),
                   'user_weighted': label('用户等权', 'Users equally weighted')}).plot.bar(
    rot=20, color=['#2563a6', '#c56a28'])
ax.set(title=label('同一词表，两种观察单位', 'Same rules, different weighting units'),
       xlabel=label('主题词类别', 'Keyword category'), ylabel=label('命中比例', 'Keyword share'))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

**怎么看结果：** 第一列回答“随机取一条已采集帖子，词命中的比例是多少”；
第二列回答“随机取一位已观测用户，其帖子命中比例的平均值是多少”。
一个只有一条记录的用户，某主题比例可能是 0 或 1，不能据此认定稳定偏好。

### 3.3 为观察用户内容结构，明确记录门槛

选取至少有 3 条记录的用户，并报告保留人数。
门槛能减少一条记录就形成 100% 比例的情况，但会改变用户集合；这不是消除采集偏差的方法。

In [ ]:
MIN_POSTS = 3
eligible_ids = users.index[users['n_posts'].ge(MIN_POSTS)]
print('至少3条记录的用户数：', len(eligible_ids), '占全部已观测用户：', len(eligible_ids) / len(users))
display(users.loc[eligible_ids, ['n_posts', 'observed_days'] +
                  ['topic_share_' + name for name in TOPICS]].head(3))
sample_food = posts.loc[topic_flags['饮食'], ['city', 'text']].head(3).copy()
sample_food['text'] = sample_food['text'].str.slice(0, 150)
display(sample_food)

**怎么看结果：** 当前有 3,295 位用户满足门槛，只覆盖原用户集合的一部分。
饮食示例是前三条可见命中记录，显示文字截到 150 字，完整文本仍用于规则计算。
看几个例子可以发现规则问题，但不能用它们声称词表准确率已经验证。
要描述“更常发什么”，应同时报告词表、用户集合、记录数及抽样核验情况。

<a id="expression"></a>
## 4．公开表达中的情绪词

### 4.1 四类可解释规则

我们使用主课相同的正向与负向词表，只做文本表达的探索。
正负都命中时保留“混合命中”，都未命中时保留“未命中”；未命中不等于中性。
规则不处理所有否定、引用、反讽或多义词，因此不能把结果当成真实心理测量。

In [ ]:
POS_WORDS = ['开心', '快乐', '幸福', '舒服', '凉快', '爽', '喜欢', '美好', '期待', '满意']
NEG_WORDS = ['难受', '烦躁', '崩溃', '痛苦', '讨厌', '失眠', '焦虑', '中暑', '热死', '受不了', '累', '郁闷', '难熬']
positive_hit = keyword_hit(posts['text'], POS_WORDS)
negative_hit = keyword_hit(posts['text'], NEG_WORDS)
emotion_labels = pd.Series(np.select(
    [positive_hit & negative_hit, negative_hit, positive_hit],
    ['混合命中', '仅负向命中', '仅正向命中'], default='未命中'), index=posts.index, name='情绪词规则')
emotion_order = ['仅正向命中', '仅负向命中', '混合命中', '未命中']
emotion_counts = emotion_labels.value_counts().reindex(emotion_order, fill_value=0)
emotion_table = pd.DataFrame({'帖子数': emotion_counts, '占全部帖子比例': emotion_counts / len(posts)})
display(emotion_table)
ax = emotion_table['占全部帖子比例'].rename(index={
    k: label(k, v) for k, v in zip(emotion_order, ['Positive only', 'Negative only', 'Mixed hits', 'No hits'])
}).plot.bar(rot=15, color=['#50896d', '#c56a28', '#8a6eaa', '#b0b8c0'])
ax.set(title=label('文本中的情绪词命中情况', 'Emotion keyword matches in text'),
       xlabel=label('规则类别', 'Rule category'), ylabel=label('占全部帖子比例', 'Share of all posts'))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

### 4.2 回到文本，看看规则命中了什么

每类展示最多两条真实记录。观察词表是否抓到了作者自己的表达，还是转述、广告或其他语境。
文本已预处理和脱敏，有些上下文可能丢失；本教程不会尝试恢复被掩码的信息。

In [ ]:
emotion_examples = posts[['city', 'text']].assign(规则=emotion_labels)
emotion_examples = emotion_examples.groupby('规则', sort=False).head(2).copy()
emotion_examples['text'] = emotion_examples['text'].str.slice(0, 150)
display(emotion_examples[['规则', 'city', 'text']])

### 4.3 文件里的源模型标签怎样使用？

文件还保留 `sentiment_label_source` 与源分数。数据说明将标签 0 定义为负向、1 定义为正向，
但具体模型版本及本数据上的语义性能没有独立核实。
`score_valid` 检查数值范围和两类分数的和，不检验模型是否正确理解了句子。
所以下面只展示来源标签数量，不把它变成用户幸福感、抑郁程度或“情绪真值”。

In [ ]:
display(posts['sentiment_label_source'].value_counts(dropna=False).sort_index()
        .rename('来源标签记录数').to_frame())
print('源分数数值有效标记占比：', posts['score_valid'].mean())
print('标签分布和数值有效率不是语义准确率。')

<a id="city"></a>
## 5．城市汇总：数量与每用户记录数

“哪个城市发帖量较多”可以先描述为“哪个来源城市标签下记录较多”。
然后同时查看不同用户数，以及该标签下每位已观测用户的平均记录数。
源城市编码的生成机制未核实，因此城市标签不是经过验证的用户居住地。
这里不分析用户跨城市移动，也不使用坐标。

In [ ]:
city_summary = posts.groupby('city').agg(
    n_posts=('post_id', 'size'), n_users=('user_id', 'nunique'))
city_summary['posts_per_observed_user'] = city_summary['n_posts'] / city_summary['n_users']
display(city_summary.sort_values('n_posts', ascending=False).head(6))
city_user_records = posts.groupby(['city', 'user_id']).size()
city_summary['median_posts_per_user'] = city_user_records.groupby(level='city').median()

count_top = city_summary['n_posts'].nlargest(6).sort_values()
# 示范门槛：避免用极小人数的城市均值直接排名；门槛不是统计显著性标准。
city_comparable = city_summary.loc[city_summary['n_users'].ge(100)]
mean_top = city_comparable['posts_per_observed_user'].nlargest(6).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
axes[0].barh(count_top.index.astype(str), count_top.to_numpy(dtype=float), color='#2563a6')
axes[1].barh(mean_top.index.astype(str), mean_top.to_numpy(dtype=float), color='#c56a28')
axes[0].set(title=label('来源城市：记录数量前六项', 'Top six city labels by post count'),
            xlabel=label('记录数', 'Collected posts'))
axes[1].set(title=label('来源城市：每用户平均记录数前六项', 'Top six city labels by mean posts per user'),
            xlabel=label('每位已观测用户的记录数', 'Posts per observed user'))
for ax in axes:
    ax.set_xlim(left=0)
    ax.grid(axis='x', alpha=0.2)
plt.show()

**怎么看结果：** 两种排名可能不同；记录数量多，可能是用户数多，并非每个用户记录更多。
“每用户记录数”仍受原采集覆盖影响，不能代表城市居民发帖倾向。
同一用户若有多个来源城市标签，会被计入多个城市的用户数，所以各城市用户数相加不一定等于全体去重用户数。
别把记录数、用户数和居民人口数混成同一分母。

<a id="questions"></a>
## 6．开放问题

每题紧接着一个**真正空白的 Code cell**，可以写自己的分析代码，也可以先讨论再展开答案。
答案只是一个有依据的分析方案，不要求与它得出同样的措辞或采用同样的门槛。
参考代码变量以 `q1_`、`q2_`、`q3_` 开头，可以复制到该题空白格运行，不覆盖上方主要统计表。
点击“运行全部”会运行示例并跳过空白格；不会自动执行折叠答案。

### 开放问题 1：怎样定义“记录较多的用户”？

如果把“窗口内至少 3 条记录”与“至少 5 条记录”分别作为门槛，
选出的用户占全部用户多少？贡献了全部记录的多少？
哪个定义更适合你的问题？这些门槛能够说明用户在平台上的真实活跃度吗？

**在下面的空白代码格写自己的比较；参考答案在其后折叠。**

<details><summary>展开参考思路与完整代码（Q1）</summary><p><strong>参考思路：</strong> 两个门槛都可以用，但必须说明它是“已采集记录数”的分组。 用户占比的分母是全体已观测用户；记录贡献占比的分母是所有采集记录。 提高门槛会缩小用户集合，不能只展示比例而不报告人数。</p><p>当前数据中至少 3 条记录的用户有 3,295 位，至少 5 条有 1,001 位；代码给出两种分母下的结果。 是否选择某门槛，要看希望覆盖哪些用户以及指标是否需要更多记录支持。 不能把未被采集到的行为当成零，也不能把这两个组直接命名为稳定的高/低活跃人群。</p><pre><code class="language-python">q1_rows = []
for q1_threshold in [3, 5]:
    q1_selected = users.loc[users[&#x27;n_posts&#x27;].ge(q1_threshold)]
    q1_rows.append({
        &#x27;记录门槛&#x27;: q1_threshold,
        &#x27;用户数&#x27;: len(q1_selected),
        &#x27;用户占比&#x27;: len(q1_selected) / len(users),
        &#x27;记录贡献数&#x27;: q1_selected[&#x27;n_posts&#x27;].sum(),
        &#x27;记录贡献占比&#x27;: q1_selected[&#x27;n_posts&#x27;].sum() / len(posts),
    })
q1_comparison = pd.DataFrame(q1_rows)
display(q1_comparison)</code></pre></details>

### 开放问题 2：“更常谈饮食”会不会只是词表定义的结果？

在现有饮食词表中加入“外卖”和“咖啡”，帖子命中比例和用户等权比例各改变多少？
新增命中的正文是否都在表达作者自己的饮食偏好？
你会展示哪些记录，说明新规则的收益与误检可能？

**在下面的空白代码格写自己的分析；参考答案在其后折叠。**

<details><summary>展开参考思路与完整代码（Q2）</summary><p><strong>参考思路：</strong> 先保持数据范围和分母不变，只改变词表，才能把变化归因于这次规则修改。 两种权重都要看，随后阅读新增命中的文本。新增词会增加或保持命中数， 但是否提高识别质量，需要语境核验，不能从命中数上升直接判断。</p><p>参考代码同时比较全体已观测用户与固定的至少 3 条记录用户集合。 小样本语境检查只用于发现问题，不用于宣称准确率。 “咖啡”可能出现在广告、引用或工作语境中，因此结论仍应表述为“饮食词命中比例”。</p><pre><code class="language-python">q2_old = topic_flags[&#x27;饮食&#x27;]
q2_words = TOPICS[&#x27;饮食&#x27;] + [&#x27;外卖&#x27;, &#x27;咖啡&#x27;]
q2_new = keyword_hit(posts[&#x27;text&#x27;], q2_words)
q2_old_users = q2_old.groupby(posts[&#x27;user_id&#x27;]).mean()
q2_new_users = q2_new.groupby(posts[&#x27;user_id&#x27;]).mean()
q2_comparison = pd.DataFrame({
    &#x27;原词表&#x27;: [q2_old.mean(), q2_old_users.mean(), q2_old_users.loc[eligible_ids].mean()],
    &#x27;扩展词表&#x27;: [q2_new.mean(), q2_new_users.mean(), q2_new_users.loc[eligible_ids].mean()],
}, index=[&#x27;全部帖子等权比例&#x27;, &#x27;全体用户等权比例&#x27;, &#x27;固定至少3条记录用户等权比例&#x27;])
q2_comparison[&#x27;变化百分点&#x27;] = (q2_comparison[&#x27;扩展词表&#x27;] - q2_comparison[&#x27;原词表&#x27;]) * 100
display(q2_comparison)
print(&#x27;新增命中记录数：&#x27;, (q2_new &amp; ~q2_old).sum())
q2_examples = posts.loc[q2_new &amp; ~q2_old, [&#x27;city&#x27;, &#x27;text&#x27;]].head(5).copy()
q2_examples[&#x27;text&#x27;] = q2_examples[&#x27;text&#x27;].str.slice(0, 150)
display(q2_examples)</code></pre></details>

### 开放问题 3：怎样定义“负向表达更多”？

“仅负向词命中”与“含负向词（包括正负混合）”是否给出相同结果？
分母用全部帖子，还是只用命中过情绪词的帖子，会有什么区别？
能否据此判断用户更不幸福或更抑郁？如果不能，你会怎样写结论？

**在下面的空白代码格写自己的比较；参考答案在其后折叠。**

<details><summary>展开参考思路与完整代码（Q3）</summary><p><strong>参考思路：</strong> 先把“负向”的定义说清楚。正负混合记录是否算入，会改变分子。 “全部帖子”与“词表覆盖到的帖子”是两个不同的分母；只用命中帖子会排除未命中记录。 也可报告用户等权结果，避免把高记录用户的表达当成所有用户的表达。</p><p>无论采用哪种定义，这些结果只是词表对已采集文本的匹配，不是用户心理状态的测量。 否定、转述、反讽与缺失上下文都可能造成误判。源模型标签也没有解决采集覆盖或心理测量问题。</p><p>可用的结论句式：<strong>“在四川两周采集范围内，按这份词表及这一定义，____比例为____； 其中词表覆盖比例为____。该结果描述公开文本中的规则命中，不能推出用户的幸福感或心理健康状态。”</strong></p><pre><code class="language-python">q3_matched = positive_hit | negative_hit
q3_negative_only = emotion_labels.eq(&#x27;仅负向命中&#x27;)
q3_any_negative = negative_hit
q3_denominator = int(q3_matched.sum())
q3_rows = []
for q3_name, q3_flag in [(&#x27;仅负向词命中&#x27;, q3_negative_only), (&#x27;含任意负向词&#x27;, q3_any_negative)]:
    q3_user_rates = q3_flag.groupby(posts[&#x27;user_id&#x27;]).mean()
    q3_rows.append({
        &#x27;定义&#x27;: q3_name,
        &#x27;命中帖子数&#x27;: int(q3_flag.sum()),
        &#x27;占全部帖子比例&#x27;: q3_flag.mean(),
        &#x27;占情绪词命中帖子比例&#x27;: q3_flag.sum() / q3_denominator if q3_denominator else np.nan,
        &#x27;全体用户等权比例&#x27;: q3_user_rates.mean(),
    })
display(pd.DataFrame(q3_rows))
print(&#x27;词表覆盖全部帖子的比例：&#x27;, q3_matched.mean())
print(&#x27;未命中帖子数：&#x27;, (~q3_matched).sum())</code></pre></details>

## 分析流程

写清研究问题 → 核对数据是否保留所需行为 → 确定帖子或用户等分析单位 → 定义指标和分母 →
查看表格与图 → 阅读真实语境 → 修改一次定义并比较 → 写出范围明确的结论。

本教程重点是描述与解释，没有因果估计、统计显著性检验或用户心理诊断。
如果进一步做统计推断，还需要根据用户重复记录、采集方式和比较目标设计分析。

读表、筛选与分组操作可查阅 [Pandas Analysis](02_pandas_analysis.ipynb)，
绘图操作可查阅 [Plotting](03_plotting.ipynb)。